# SPIQ MaxCut Workflow

End-to-end SPIQ initialization on MaxCut, followed by two point-selection strategies for multi-start optimization.

In [1]:
import warnings

import numpy as np
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from spiq.graphs import build_max_cut_paulis, generate_k_regular_graph
from spiq.qaoa import QAOASolver
from spiq.selection import fixed_interval_selection, k_gaps_selection

In [2]:
N_QUBITS = 10
REPS = 2
N_GENS = 100
SEED = 0
NUM_SELECT = 3

np.random.seed(SEED)

graph = generate_k_regular_graph(
    num_vertices=N_QUBITS, k=3, weighted=False, seed=SEED
)
cost_hamiltonian = SparsePauliOp.from_list(build_max_cut_paulis(graph))
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=REPS)

solver = QAOASolver(cost_hamiltonian, circuit, sim_device="CPU")
solver.err = None
solver.prepare_circuit()
exact_energy = solver.evaluate_exact_energy()
print(f"qubits={N_QUBITS}, reps={REPS}, params={solver.pcirc.num_parameters}")
print(f"exact energy={exact_energy:.6f}")

Exact Energy from Eigensolver: -11.0
qubits=10, reps=2, params=50
exact energy=-11.000000


In [3]:
solver.run_spiq(n_gens=N_GENS)

best_params = solver.best_spiq_gen_params[::-1]
best_fitness = solver.best_spiq_gen_fitness[::-1]
print(f"SPIQ best energy={solver.energy_best:.6f}")
print(f"candidate points={len(best_fitness)}")

STARTING ROUND 0


started GA at id None with 2 procs

GA parameters used for this experiment:
  num_generations=50
  num_parents_mating=20
  population_size=100
  num_genes=50
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5
GENERATION 1: BEST FITNESS FOUND : -2.0
GENERATION 2: BEST FITNESS FOUND : -3.0
GENERATION 7: BEST FITNESS FOUND : -4.0
GENERATION 13: BEST FITNESS FOUND : -5.0
GENERATION 18: BEST FITNESS FOUND : -7.0
GENERATION 32: BEST FITNESS FOUND : -8.0
Minimum Energy found with SPIQ initialization: -8.0
SPIQ best energy=-8.000000
candidate points=51


In [4]:
spaced_params, spaced_fitness = fixed_interval_selection(
    best_params, best_fitness, num_select=NUM_SELECT
)
for i, (params, energy) in enumerate(zip(spaced_params, spaced_fitness)):
    print(f"fixed interval {i + 1}: energy={energy:.6f}, params={params}")

fixed interval 1: energy=-8.000000, params=[0 0 3 1 2 0 2 3 2 1 1 3 1 0 0 3 3 0 1 3 2 0 3 1 0 2 3 0 2 0 3 3 0 2 2 3 0
 2 2 0 0 1 1 1 3 1 1 1 2 1]
fixed interval 2: energy=-4.000000, params=[1 2 1 1 0 2 2 1 2 1 0 3 1 0 2 3 3 0 0 3 2 0 3 1 0 2 3 0 2 0 3 3 0 2 2 3 0
 0 2 0 0 1 0 1 2 1 1 0 2 1]


In [5]:
kgaps_result = k_gaps_selection(
    best_params,
    best_fitness,
    solver,
    num_select=NUM_SELECT,
    rng=np.random.default_rng(SEED),
)
kgaps_result

(array([[1, 2, 1, 0, 2, 2, 3, 2, 1, 1, 3, 1, 1, 2, 3, 0, 0, 2, 2, 0, 0, 3,
         2, 0, 1, 3, 2, 0, 2, 0, 3, 2, 0, 0, 3, 2, 1, 2, 0, 1, 3, 1, 2, 3,
         1, 2, 3, 0, 2, 0],
        [2, 2, 2, 2, 2, 1, 2, 3, 2, 3, 2, 2, 2, 0, 3, 3, 2, 1, 3, 2, 3, 3,
         3, 0, 2, 2, 3, 0, 2, 3, 1, 0, 2, 2, 0, 3, 2, 3, 1, 1, 3, 0, 0, 1,
         1, 1, 1, 0, 2, 2],
        [1, 0, 1, 0, 3, 2, 0, 2, 0, 3, 1, 2, 3, 0, 3, 3, 3, 1, 1, 0, 0, 1,
         3, 2, 2, 1, 0, 2, 2, 0, 1, 0, 3, 0, 3, 2, 2, 3, 1, 0, 1, 1, 3, 1,
         2, 0, 0, 1, 0, 2]]),
 array([-3., -3., -2.]),
 array([2.23606798, 2.44948974, 2.44948974]))